# NYC Yellow Taxi — Trip Quality Analysis 2018–2022
### Stratified Sample · ~289,000 trips (post-cleaning) · Polars + DuckDB

---

## Management Summary

**What this data is:** ~305,000 NYC TLC Yellow Taxi trips sampled equally across
2018–2022 (≈60k per year), plus a partial January 2023 slice. Sampling equalises
trip count **per year**, so this cannot measure absolute ridership volumes. It
measures how the *shape* of a typical trip changed — fare levels, payment
behaviour, speed, and trip-type mix.

---

### Key Findings and Operational Implications

**1. Total ride cost up ~30%** ($16.39 → $21.28), but the rise is concentrated
in two line items: the congestion surcharge ($2.34/trip, 48% of the increase)
and higher tips ($0.83/trip, 17%). Base fare growth contributed only $1.38.  
*→ Any cost or elasticity model using `fare_amount` alone understates the rider's
true bill by 15–20% in post-2019 data.*

**2. Congestion surcharge: from zero to structural** (0% → 93%+ of trips by 2021).
Riders absorbed the $2.50 charge without visible defection over the sample period.  
*→ It is permanent baseline, not a recoverable surcharge. Competitive comparisons
with TNCs must include it; omitting it overstates the price gap.*

**3. Cash near-halved** (30% → 20% of trips). Lower cash-handling costs for
operators, but the remaining 20% likely skews toward unbanked, tourist, or
privacy-motivated riders whose trip profile differs from card payers.  
*→ Eliminating cash would be a non-trivial access policy decision, not a clean
operational simplification.*

**4. In-app tip defaults drove a 5pp rate increase** (22% → 27% on card trips).
The mechanism is prompt design — the 20–25% bucket collapsed as app presets
shifted upward — not greater rider generosity. This delivered ~$0.74/trip more
to drivers without any fare change.  
*→ An equivalent income gain via base fare hike would require a ~5% rate increase,
which carries demand-side risk. Prompt design does not.*

**5. 2020 speed anomaly: +16%** (11.3 → 13.1 mph average). Emptied streets moved
trips above the TLC meter's 12 mph time-rate threshold, lowering fare per mile
even as trip cost fell overall. The improvement was largest at rush hours,
confirming a structural congestion collapse rather than a trip-type shift.  
*→ Fleet capacity models should treat 2020-level speeds as a low-demand ceiling,
not a recoverable norm.*

**6. Airport trip share rose ~65%** (2.4% → 4.0% in 2022, from a small base).
Consistent with post-pandemic air travel recovery. TNCs still dominate airport
ground transport; this finding rests on ~1,400–2,300 sample trips/year and should
be treated as a directional signal, not a precise market share estimate.  
*→ A borough-level and route-level cut of the full TLC public dataset would be
needed to size this opportunity reliably.*

---

### Sampling Design Caveat

Each year contributes ~60,000 trips. Cross-year comparisons of *rates* (fare/mile,
tip %, payment mix) are valid.
Comparisons of *absolute volume* are **not** — 2020 had far fewer actual rides,
but the sample forces parity.
Within-year monthly distributions are preserved; the COVID dip is confirmed visible
inside the 2020 slice (§ 6).

---

*Full methodology and charts follow. Analysis excludes the partial January 2023 slice.*


## 1. Setup & Ingestion

The CSV is read once with **Polars** (zero-copy columnar frame), then queried
throughout via **DuckDB** — which resolves the Python variable name directly,
giving us SQL over a Polars frame with no serialisation overhead.

`infer_schema_length=100_000` is required because `airport_fee` and
`congestion_surcharge` are empty in the first several thousand rows (those fees
did not exist in 2018), causing the default 100-row inferrer to mis-type them.


In [ ]:
import polars as pl
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ── colour palette ────────────────────────────────────────────────────────────
BLUE   = '#3D7EAA'
GREEN  = '#4CAF78'
ORANGE = '#E8833A'
RED    = '#D94F3D'
PURPLE = '#7B5EA7'
GREY   = '#AAAAAA'

plt.rcParams.update({
    'figure.dpi':        130,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'axes.grid.axis':    'y',
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
})

# Resolve data path whether CWD is project root (jupyter lab) or notebooks/ (nbconvert)
from pathlib import Path
_data = Path('data/Yellow_Taxi_Assignment.csv')
if not _data.exists():
    _data = Path('../data/Yellow_Taxi_Assignment.csv')
CSV_PATH = _data
trips_raw = pl.read_csv(CSV_PATH, infer_schema_length=100_000, try_parse_dates=True)
print(f'Loaded  {trips_raw.shape[0]:,} rows × {trips_raw.shape[1]} columns')
trips_raw.head(3)


## 2. Data Quality Audit & Cleaning

### Null landscape

Fourteen of 19 columns are fully populated. The three structurally incomplete
columns, and a fourth that carries negative values, are documented below before
any rows are removed. The audit code cell that follows produces all counts as
queryable output.

#### Structural null block — 3.12% of rows
The 9,513 rows where `payment_type = 0` are *simultaneously* null in
`passenger_count`, `RatecodeID`, and `store_and_fwd_flag`. These are not
independent missing values — they are the same incomplete dispatch records.
The `payment_type != 0` drop rule clears the entire block in one step; no
separate null imputation is needed for those three columns.

#### Fee columns — structural absence, not missing data
| Column | Null % | Root cause | Decision |
|--------|--------|-----------|----------|
| `airport_fee` | 65.2% | `VARCHAR`; fee introduced ~2022; pre-2022 rows are empty string | Cast to `DOUBLE`; null / empty → 0; 55 rows of `"-1.25"` (refund artefacts) floored to 0 via `GREATEST(..., 0)` |
| `congestion_surcharge` | 23.8% | Fee did not exist before Feb 2019; also exempt outer-borough trips | Null → 0 (structural absence, not missing data) |
| `passenger_count` | 3.12% | Same block as `payment_type = 0` records | Block dropped; no imputation required |

### Validity rules — rows dropped outright

| Rule | Rationale |
|------|-----------|
| `fare_amount > 0` | Negative fares are refund artefacts or encoding errors |
| `total_amount > 0` | Same — a few hundred rows have negative totals |
| `trip_distance > 0` | GPS / sensor failures |
| `trip_distance ≤ 200 mi` | A 177,247-mile row is a sensor blowout; NYC max is ~100 mi |
| `dropoff > pickup` | 400 timestamp reversals — likely clock issues |
| `RatecodeID ≠ 99` | Code 99 = "unknown"; too rare (<300 rows) to impute |
| `payment_type ≠ 0` | Undocumented code; clears the 3.12% null block simultaneously |

### Negative ancillary charges — retained with documentation

Six fee columns contain small numbers of negative values that **pass** the
validity rules (because `total_amount` is still positive on those rows):

| Column | Negative rows | Typical value | Interpretation |
|--------|--------------|---------------|----------------|
| `improvement_surcharge` | 1,160 (0.38%) | −$0.30 | Voided / refunded charge |
| `mta_tax` | 1,123 (0.37%) | −$0.50 | Voided / refunded charge |
| `congestion_surcharge` | 891 (0.29%) | −$2.50 | Voided / refunded charge |
| `extra` | 541 (0.18%) | −$0.50 | Voided rush-hour surcharge |
| `tolls_amount` | 38 (0.01%) | varies | Refund |
| `tip_amount` | 14 (< 0.01%) | varies | Tip reversal |

**Decision: retain.** These are real financial corrections on otherwise valid trip
records. Their effect on per-component averages is < $0.002 / trip — negligible
for analysis. Any downstream aggregation of raw components should be aware of them.

### Undocumented categoricals

| Column | Undocumented value(s) | Count | Treatment |
|--------|-----------------------|-------|-----------|
| `VendorID` | 4, 5, 6 | 1,024 rows | Retained; excluded from any vendor-specific cut |
| `RatecodeID` | 6 | 1 row | Retained; falls into "Other" trip-type category |
| `store_and_fwd_flag` | Y (documented but rarely flagged) | 3,040 (1.0%) | Y = record held in vehicle memory before upload. Retained; not used in analysis |


In [ ]:
# ── 1. Per-column null audit ─────────────────────────────────────────────────
null_audit = duckdb.sql('''
    SELECT col, null_count, total,
           ROUND(100.0 * null_count / total, 2) AS null_pct,
           dtype
    FROM (
        VALUES
          ('airport_fee',           (SELECT COUNT(*) FILTER (WHERE airport_fee IS NULL)           FROM trips_raw), 304978, 'VARCHAR → DOUBLE'),
          ('congestion_surcharge',  (SELECT COUNT(*) FILTER (WHERE congestion_surcharge IS NULL)  FROM trips_raw), 304978, 'DOUBLE'),
          ('passenger_count',       (SELECT COUNT(*) FILTER (WHERE passenger_count IS NULL)       FROM trips_raw), 304978, 'DOUBLE → INT'),
          ('RatecodeID',            (SELECT COUNT(*) FILTER (WHERE RatecodeID IS NULL)            FROM trips_raw), 304978, 'DOUBLE → INT'),
          ('store_and_fwd_flag',    (SELECT COUNT(*) FILTER (WHERE store_and_fwd_flag IS NULL)    FROM trips_raw), 304978, 'VARCHAR'),
          ('fare_amount',           (SELECT COUNT(*) FILTER (WHERE fare_amount IS NULL)           FROM trips_raw), 304978, 'DOUBLE'),
          ('total_amount',          (SELECT COUNT(*) FILTER (WHERE total_amount IS NULL)          FROM trips_raw), 304978, 'DOUBLE'),
          ('tip_amount',            (SELECT COUNT(*) FILTER (WHERE tip_amount IS NULL)            FROM trips_raw), 304978, 'DOUBLE'),
          ('trip_distance',         (SELECT COUNT(*) FILTER (WHERE trip_distance IS NULL)         FROM trips_raw), 304978, 'DOUBLE'),
          ('VendorID',              (SELECT COUNT(*) FILTER (WHERE VendorID IS NULL)              FROM trips_raw), 304978, 'BIGINT'),
          ('tpep_pickup_datetime',  (SELECT COUNT(*) FILTER (WHERE tpep_pickup_datetime IS NULL)  FROM trips_raw), 304978, 'TIMESTAMP'),
          ('tpep_dropoff_datetime', (SELECT COUNT(*) FILTER (WHERE tpep_dropoff_datetime IS NULL) FROM trips_raw), 304978, 'TIMESTAMP')
    ) t(col, null_count, total, dtype)
    ORDER BY null_count DESC
''').pl()

print('── Null audit ──')
display(null_audit)

# ── 2. Negative values in ancillary fee columns ───────────────────────────────
neg_audit = duckdb.sql('''
    SELECT col, neg_count,
           ROUND(100.0 * neg_count / 304978, 3) AS pct_of_raw
    FROM (
        VALUES
          ('improvement_surcharge', (SELECT COUNT(*) FROM trips_raw WHERE improvement_surcharge < 0)),
          ('mta_tax',               (SELECT COUNT(*) FROM trips_raw WHERE mta_tax               < 0)),
          ('congestion_surcharge',  (SELECT COUNT(*) FROM trips_raw WHERE congestion_surcharge  < 0)),
          ('extra',                 (SELECT COUNT(*) FROM trips_raw WHERE extra                 < 0)),
          ('tolls_amount',          (SELECT COUNT(*) FROM trips_raw WHERE tolls_amount          < 0)),
          ('tip_amount',            (SELECT COUNT(*) FROM trips_raw WHERE tip_amount            < 0))
    ) t(col, neg_count)
    ORDER BY neg_count DESC
''').pl()

print('\n── Negative ancillary charges (pass validity filter; retained) ──')
display(neg_audit)

# ── 3. airport_fee raw string values (VARCHAR column) ────────────────────────
fee_vals = duckdb.sql('''
    SELECT COALESCE(airport_fee, 'NULL') AS raw_value,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / 304978, 2) AS pct
    FROM trips_raw
    GROUP BY airport_fee
    ORDER BY n DESC
''').pl()

print('\n── airport_fee raw values (VARCHAR) ──')
display(fee_vals)

# ── 4. VendorID — confirm undocumented codes ──────────────────────────────────
vendor_vals = duckdb.sql('''
    SELECT VendorID,
           CASE VendorID
               WHEN 1 THEN 'Creative Mobile Technologies'
               WHEN 2 THEN 'VeriFone Inc.'
               ELSE        'Undocumented'
           END AS vendor_name,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / 304978, 2) AS pct
    FROM trips_raw
    GROUP BY VendorID
    ORDER BY n DESC
''').pl()

print('\n── VendorID values ──')
display(vendor_vals)


In [ ]:
trips_clean = duckdb.sql('''
    WITH casted AS (
        SELECT
            *,
            -- airport_fee is VARCHAR; 55 rows hold "-1.25" (refund artefacts).
            -- GREATEST(..., 0) floors those to 0 after the numeric cast.
            GREATEST(TRY_CAST(airport_fee AS DOUBLE), 0) AS airport_fee_num
        FROM trips_raw
    ),
    filtered AS (
        SELECT * FROM casted
        WHERE  fare_amount                        >  0
           AND total_amount                       >  0
           AND trip_distance                      >  0
           AND trip_distance                      <= 200
           AND tpep_dropoff_datetime              >  tpep_pickup_datetime
           AND ratecodeid                         != 99
           AND payment_type                       != 0
    )
    SELECT
        VendorID                                  AS vendor_id,
        tpep_pickup_datetime                      AS pickup_dt,
        tpep_dropoff_datetime                     AS dropoff_dt,
        COALESCE(passenger_count, 1)::INT         AS passenger_count,
        trip_distance,
        ratecodeid::INT                           AS rate_code,
        store_and_fwd_flag,
        PULocationID                              AS pu_location_id,
        DOLocationID                              AS do_location_id,
        payment_type::INT                         AS payment_type,
        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        total_amount,
        COALESCE(congestion_surcharge, 0)         AS congestion_surcharge,
        COALESCE(airport_fee_num,      0)         AS airport_fee
    FROM filtered
''').pl()

n_raw   = trips_raw.shape[0]
n_clean = trips_clean.shape[0]
n_drop  = n_raw - n_clean
print(f'Raw rows  : {n_raw:,}')
print(f'Dropped   : {n_drop:,}  ({100*n_drop/n_raw:.1f}%)')
print(f'Clean rows: {n_clean:,}')


In [ ]:
duckdb.sql('''
    SELECT rule, violations,
           ROUND(100.0 * violations / (SELECT COUNT(*) FROM trips_raw), 2) AS pct_of_raw
    FROM (
        VALUES
          ('Negative / zero fare',         (SELECT COUNT(*) FROM trips_raw WHERE fare_amount  <= 0)),
          ('Negative / zero total',         (SELECT COUNT(*) FROM trips_raw WHERE total_amount <= 0)),
          ('Non-positive distance',         (SELECT COUNT(*) FROM trips_raw WHERE trip_distance <= 0)),
          ('Distance > 200 mi',             (SELECT COUNT(*) FROM trips_raw WHERE trip_distance > 200)),
          ('Dropoff <= pickup time',        (SELECT COUNT(*) FROM trips_raw WHERE tpep_dropoff_datetime <= tpep_pickup_datetime)),
          ('RateCode 99 (unknown)',          (SELECT COUNT(*) FROM trips_raw WHERE ratecodeid = 99)),
          ('Payment type 0 (untracked)',    (SELECT COUNT(*) FROM trips_raw WHERE payment_type = 0))
    ) t(rule, violations)
    ORDER BY violations DESC
''').pl()


## 3. Feature Engineering

Derived columns added on top of the clean frame. Two additional
physical-plausibility guards trim extreme derived values that survive
the raw cleaning pass:

| Feature | Formula | Range guard |
|---------|---------|-------------|
| `pickup_year` / `pickup_month` / `pickup_hour` | date parts from `pickup_dt` | — |
| `duration_min` | dropoff − pickup in minutes | 1–240 min |
| `fare_per_mile` | fare_amount ÷ trip_distance | $0.50–$50 |
| `speed_mph` | distance ÷ (duration / 60) | 0–80 mph |
| `tip_pct` | tip_amount ÷ fare_amount × 100 | NULL for zero-fare rows |
| `payment_label` | 1 → Card · 2 → Cash · else → Other | — |
| `trip_type` | rate_code 2/3 → Airport · 1 → Standard | — |
| `has_congestion` | congestion_surcharge > 0 | Boolean flag |


In [ ]:
trips_fe = duckdb.sql('''
    SELECT
        *,
        YEAR(pickup_dt)                                                               AS pickup_year,
        MONTH(pickup_dt)                                                              AS pickup_month,
        HOUR(pickup_dt)                                                               AS pickup_hour,
        DATEDIFF('minute', pickup_dt, dropoff_dt)                                    AS duration_min,
        ROUND(fare_amount / trip_distance, 2)                                        AS fare_per_mile,
        ROUND(trip_distance / (DATEDIFF('minute', pickup_dt, dropoff_dt) / 60.0), 1) AS speed_mph,
        ROUND(tip_amount / NULLIF(fare_amount, 0) * 100, 1)                          AS tip_pct,
        CASE payment_type
            WHEN 1 THEN 'Card'
            WHEN 2 THEN 'Cash'
            ELSE        'Other'
        END                                                                           AS payment_label,
        CASE
            WHEN rate_code IN (2, 3) THEN 'Airport'
            WHEN rate_code  = 1      THEN 'Standard'
            ELSE                          'Other'
        END                                                                           AS trip_type,
        CASE WHEN congestion_surcharge > 0 THEN TRUE ELSE FALSE END                  AS has_congestion
    FROM trips_clean
    WHERE DATEDIFF('minute', pickup_dt, dropoff_dt) BETWEEN 1 AND 240
      AND fare_amount / trip_distance BETWEEN 0.5 AND 50
      AND trip_distance / (DATEDIFF('minute', pickup_dt, dropoff_dt) / 60.0) BETWEEN 0 AND 80
''').pl()

print(f'Analysis frame: {trips_fe.shape[0]:,} rows × {trips_fe.shape[1]} columns')
print(f'Extreme-value rows removed by derived filters: {trips_clean.shape[0] - trips_fe.shape[0]:,}')

duckdb.sql('''
    SELECT pickup_year, COUNT(*) AS trips
    FROM trips_fe
    GROUP BY pickup_year
    ORDER BY pickup_year
''').pl()


## 4. Fare Economics

**Total ride cost grew ~30%** from $16.39 (2018) to $21.28 (2022). The
congestion surcharge — introduced February 2019 at $2.50 per Manhattan trip —
accounts for 48% of that rise. See § 11 for a full dollar-by-dollar decomposition.

**The `fare_per_mile` U-shape has a specific mechanical explanation.** The TLC
meter operates on two modes: $0.70 per ⅕ mile when the vehicle moves *faster*
than 12 mph (distance rate), or $0.70 per 60 seconds when moving *at or below*
12 mph (time rate). In 2018–2019, average trip speed was 11.3–11.4 mph — just
*below* that threshold — so the time rate applied frequently, inflating fare per
unit of distance. In 2020, emptied streets pushed average speed to 13.1 mph,
tipping most trips into distance-only billing and *reducing* fare per mile
($6.06 → $5.52) even though trips became shorter overall. The partial recovery
to $5.63 in 2022 reflects both the return of congestion and TLC rate adjustments.

This is not a paradox: **under the TLC formula, slower traffic literally costs
more per mile.** The pandemic made trips faster *and* cheaper per unit distance —
two effects that partially offset each other in the total fare.


In [ ]:
yoy = duckdb.sql('''
    WITH base AS (
        SELECT
            pickup_year,
            AVG(fare_per_mile)                 AS fare_per_mile,
            AVG(total_amount)                  AS avg_total,
            AVG(fare_amount)                   AS avg_fare,
            AVG(congestion_surcharge)          AS avg_congestion,
            AVG(tip_amount)                    AS avg_tip,
            AVG(tolls_amount)                  AS avg_tolls,
            AVG(extra)                         AS avg_extra,
            AVG(mta_tax)                       AS avg_mta,
            AVG(improvement_surcharge)         AS avg_improvement
        FROM trips_fe
        WHERE pickup_year < 2023
        GROUP BY pickup_year
    )
    SELECT
        pickup_year,
        ROUND(fare_per_mile,   2) AS fare_per_mile,
        ROUND(avg_fare,        2) AS avg_fare,
        ROUND(avg_congestion,  2) AS avg_congestion,
        ROUND(avg_tip,         2) AS avg_tip,
        ROUND(avg_tolls,       2) AS avg_tolls,
        ROUND(avg_extra,       2) AS avg_extra,
        ROUND(avg_mta,         2) AS avg_mta,
        ROUND(avg_improvement, 2) AS avg_improvement,
        ROUND(avg_total,       2) AS avg_total
    FROM base
    ORDER BY pickup_year
''').pl()

display(yoy)

years    = yoy['pickup_year'].to_list()
arr_fare = yoy['avg_fare'].to_list()
arr_cong = yoy['avg_congestion'].to_list()
arr_tip  = yoy['avg_tip'].to_list()
arr_ext  = yoy['avg_extra'].to_list()
arr_toll = yoy['avg_tolls'].to_list()
arr_mta  = yoy['avg_mta'].to_list()
arr_imp  = yoy['avg_improvement'].to_list()
arr_fpm  = yoy['fare_per_mile'].to_list()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── left: stacked cost components ──────────────────────────────────────────
ax = axes[0]
bottom = np.zeros(len(years))
components = [
    (arr_fare, 'Base Fare',    BLUE),
    (arr_cong, 'Congestion',   ORANGE),
    (arr_tip,  'Tip',          GREEN),
    (arr_ext,  'Extra / Rush', RED),
    (arr_toll, 'Tolls',        PURPLE),
    (arr_mta,  'MTA Tax',      GREY),
    (arr_imp,  'Improvement',  '#CCCCCC'),
]
for vals, lbl, col in components:
    ax.bar(years, vals, bottom=bottom, label=lbl, color=col, width=0.6)
    bottom += np.array(vals)
for y, tot in zip(years, bottom):
    ax.text(y, tot + 0.15, f'${tot:.2f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Average Trip Cost — Components')
ax.set_ylabel('USD')
ax.set_xticks(years)
ax.legend(fontsize=8, loc='upper left', ncol=2)

# ── right: fare per mile ───────────────────────────────────────────────────
ax2 = axes[1]
bars = ax2.bar(years, arr_fpm, color=BLUE, alpha=0.85, width=0.6)
ax2.bar_label(bars, fmt='$%.2f', padding=3, fontsize=10)
ax2.set_title('Fare per Mile')
ax2.set_ylabel('USD / mile')
ax2.set_xticks(years)
ax2.set_ylim(0, max(arr_fpm) * 1.3)

plt.suptitle('§ 4  Fare Economics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Trip Shape — Distance, Duration, and Speed

**The 2020 signal is unmistakeable.** Average trip duration fell from 14.6 min
to 11.7 min (−20%), and average speed jumped from 11.3 mph to 13.1 mph (+16%).
These are not driven by shorter trips — median distance barely changed
(1.64 mi in 2018 vs 1.69 mi in 2020). Empty streets simply accelerated
every trip.

By 2022, trips were longer than at any prior year (median 1.90 mi, avg 3.50 mi),
consistent with more airport runs and longer post-lockdown commutes, and duration
climbed to 15.6 min as traffic returned.


In [ ]:
dist_q = duckdb.sql('''
    SELECT
        pickup_year,
        ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY trip_distance), 2) AS p25_dist,
        ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY trip_distance), 2) AS med_dist,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY trip_distance), 2) AS p75_dist,
        ROUND(AVG(duration_min), 1)  AS avg_dur,
        ROUND(AVG(speed_mph),    1)  AS avg_speed
    FROM trips_fe
    WHERE pickup_year < 2023
    GROUP BY pickup_year
    ORDER BY pickup_year
''').pl()

display(dist_q)

years = dist_q['pickup_year'].to_list()
p25   = dist_q['p25_dist'].to_list()
med   = dist_q['med_dist'].to_list()
p75   = dist_q['p75_dist'].to_list()
dur   = dist_q['avg_dur'].to_list()
spd   = dist_q['avg_speed'].to_list()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# distance IQR + median
ax = axes[0]
iqr = [p75[i] - p25[i] for i in range(len(years))]
ax.bar(years, iqr, bottom=p25, color=BLUE, alpha=0.5, width=0.6, label='IQR (25–75th %ile)')
ax.plot(years, med, 'o-', color=RED, lw=2, ms=6, label='Median')
for y, m in zip(years, med):
    ax.text(y, m + 0.06, str(m), ha='center', va='bottom', fontsize=8)
ax.set_title('Trip Distance — Distribution')
ax.set_ylabel('Miles')
ax.set_xticks(years)
ax.legend(fontsize=9)

# avg duration
ax2 = axes[1]
bars2 = ax2.bar(years, dur, color=GREEN, alpha=0.85, width=0.6)
ax2.bar_label(bars2, fmt='%.1f min', padding=3, fontsize=9)
ax2.set_title('Avg Trip Duration')
ax2.set_ylabel('Minutes')
ax2.set_xticks(years)
ax2.set_ylim(0, max(dur) * 1.3)

# avg speed — highlight 2020
ax3 = axes[2]
bar_colors = [RED if y == 2020 else ORANGE for y in years]
bars3 = ax3.bar(years, spd, color=bar_colors, alpha=0.85, width=0.6)
ax3.bar_label(bars3, fmt='%.1f mph', padding=3, fontsize=9)
ax3.set_title('Avg Trip Speed')
ax3.set_ylabel('mph')
ax3.set_xticks(years)
ax3.set_ylim(0, max(spd) * 1.35)
idx = years.index(2020)
ax3.annotate('COVID: empty streets',
    xy=(2020, spd[idx]), xytext=(2021.3, spd[idx] + 0.4),
    arrowprops=dict(arrowstyle='->', color='grey'),
    fontsize=9, color='grey')

plt.suptitle('§ 5  Trip Shape', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. The 2020 COVID Shock

Because the sample draws ~60,000 trips from the full-year 2020 pool, and that
pool was near-empty in March–June 2020, the sampled monthly counts reflect the
real demand collapse proportionally.

The heatmap below shows sampled trip count per month per year. Every 2018–2019
row is nearly uniform. In 2020, April and May are visibly lighter, confirming
the COVID dip is preserved in the sampled data.


In [ ]:
monthly_q = duckdb.sql('''
    SELECT pickup_year, pickup_month, COUNT(*) AS trips
    FROM trips_fe
    WHERE pickup_year BETWEEN 2018 AND 2022
    GROUP BY pickup_year, pickup_month
    ORDER BY pickup_year, pickup_month
''').pl()

pivot = monthly_q.to_pandas().pivot(
    index='pickup_year', columns='pickup_month', values='trips'
)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
pivot.columns = month_labels

fig, ax = plt.subplots(figsize=(12, 3.5))
vmin, vmax = 3800, 5100
im = ax.imshow(pivot.values, aspect='auto', cmap='Blues', vmin=vmin, vmax=vmax)

ax.set_xticks(range(12))
ax.set_xticklabels(month_labels, fontsize=10)
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels(pivot.index.astype(str), fontsize=10)

for r in range(pivot.shape[0]):
    for c in range(pivot.shape[1]):
        v = int(pivot.values[r, c])
        brightness = (v - vmin) / (vmax - vmin)
        txt_color  = 'white' if brightness < 0.45 else 'black'
        ax.text(c, r, f'{v:,}', ha='center', va='center',
                fontsize=8, color=txt_color)

plt.colorbar(im, ax=ax, label='Sampled trips', shrink=0.9)
ax.set_title('§ 6  Monthly Trip Count by Year  (lighter = fewer trips)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Payment Type Evolution

Cash represented ~30% of trips in 2018. By 2022 it had fallen to ~20% — a
one-third reduction in five years. The shift accelerated in 2020 as contactless
payment became preferred for hygiene reasons. The "Other" category
(payment_type ≥ 3) remains flat at ~0.5%.


In [ ]:
pay_q = duckdb.sql('''
    WITH b AS (
        SELECT pickup_year, payment_label, COUNT(*) AS n
        FROM trips_fe WHERE pickup_year < 2023
        GROUP BY pickup_year, payment_label
    ),
    t AS (SELECT pickup_year, SUM(n) AS tot FROM b GROUP BY pickup_year)
    SELECT b.pickup_year, b.payment_label,
           ROUND(100.0 * b.n / t.tot, 1) AS pct
    FROM b JOIN t USING (pickup_year)
    ORDER BY b.pickup_year, b.payment_label
''').pl()

display(pay_q.to_pandas().pivot(
    index='pickup_year', columns='payment_label', values='pct'))

years  = sorted(pay_q['pickup_year'].unique().to_list())
labels = ['Card', 'Cash', 'Other']
colors = [BLUE, ORANGE, GREY]

data = {}
for lbl in labels:
    data[lbl] = [
        float(pay_q.filter(
            (pl.col('pickup_year') == y) & (pl.col('payment_label') == lbl)
        ).select('pct').to_series()[0])
        for y in years
    ]

fig, ax = plt.subplots(figsize=(8, 5))
bottom = np.zeros(len(years))
for lbl, col in zip(labels, colors):
    vals = np.array(data[lbl])
    ax.bar(years, vals, bottom=bottom, label=lbl, color=col, alpha=0.9, width=0.6)
    for y, v, b in zip(years, vals, bottom):
        if v > 3:
            ax.text(y, b + v / 2, f'{v:.0f}%',
                    ha='center', va='center', fontsize=10,
                    color='white', fontweight='bold')
    bottom += vals

ax.set_ylim(0, 112)
ax.set_ylabel('Share of trips (%)')
ax.set_xticks(years)
ax.set_yticks(range(0, 101, 20))
ax.legend(fontsize=11)
ax.set_title('§ 7  Payment Type Evolution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Tipping Behavior

Tip data is reliable only for **card trips** — cash tips are self-reported to
the meter and overwhelmingly appear as $0. This section uses card-paying
riders only.

**The shift is structural, not economic.** In 2018, nearly 49% of card trips
tipped in the 20–25% range, consistent with a default in-app prompt of 20%.
By 2022, the 25–30% and 30%+ buckets together account for 60% of card trips,
while the 20–25% bucket collapsed to 19%. This mirrors the industry-wide shift
to higher default tip presets that began around 2019–2020.

Average card tip: $2.70 (2018) → $3.44 (2022), tracking the higher default
percentages and higher base fares.


In [ ]:
tip_q = duckdb.sql('''
    WITH buckets AS (
        SELECT
            pickup_year,
            CASE
                WHEN tip_pct = 0       THEN '0%'
                WHEN tip_pct < 15      THEN '< 15%'
                WHEN tip_pct < 20      THEN '15-20%'
                WHEN tip_pct < 25      THEN '20-25%'
                WHEN tip_pct < 30      THEN '25-30%'
                ELSE                        '30%+'
            END AS bucket,
            COUNT(*) AS n
        FROM trips_fe
        WHERE pickup_year < 2023 AND payment_type = 1
        GROUP BY pickup_year, bucket
    ),
    tot AS (SELECT pickup_year, SUM(n) AS total FROM buckets GROUP BY pickup_year)
    SELECT
        b.pickup_year, b.bucket,
        ROUND(100.0 * b.n / t.total, 1) AS pct
    FROM buckets b JOIN tot t USING (pickup_year)
    ORDER BY b.pickup_year,
        CASE b.bucket
            WHEN '0%'    THEN 0 WHEN '< 15%' THEN 1
            WHEN '15-20%' THEN 2 WHEN '20-25%' THEN 3
            WHEN '25-30%' THEN 4 ELSE 5
        END
''').pl()

years   = sorted(tip_q['pickup_year'].unique().to_list())
buckets = ['0%', '< 15%', '15-20%', '20-25%', '25-30%', '30%+']
bcolors = ['#DDDDDD', '#AAAAAA', PURPLE, BLUE, GREEN, ORANGE]

data = {}
for bk in buckets:
    data[bk] = [
        float(tip_q.filter(
            (pl.col('pickup_year') == y) & (pl.col('bucket') == bk)
        ).select('pct').to_series()[0])
        if tip_q.filter(
            (pl.col('pickup_year') == y) & (pl.col('bucket') == bk)
        ).shape[0] > 0 else 0.0
        for y in years
    ]

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(years))
for bk, col in zip(buckets, bcolors):
    vals = np.array(data[bk])
    ax.bar(years, vals, bottom=bottom, label=bk, color=col, alpha=0.9, width=0.6)
    for y, v, b in zip(years, vals, bottom):
        if v > 5:
            ax.text(y, b + v / 2, f'{v:.0f}%',
                    ha='center', va='center', fontsize=8.5,
                    color='white', fontweight='bold')
    bottom += vals

ax.set_ylim(0, 112)
ax.set_ylabel('Share of card trips (%)')
ax.set_xticks(years)
ax.set_yticks(range(0, 101, 20))
ax.legend(fontsize=9, title='Tip %', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_title('§ 8  Card Tip Rate Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Airport vs. City Trips

Rate code 2 (JFK flat-rate) and rate code 3 (Newark) identify airport-bound
Yellow Taxi trips. These are already a small minority of Yellow Taxi volume —
~2.4% in 2018 — because TNC platforms dominate airport ground transport.

The COVID dip in this dataset is minor (2.4% → 2.2%), in contrast to the wider
airport industry collapse. More notable is the post-pandemic recovery to 4.0%
in 2022 — a **~65% relative increase** from the 2018–2019 baseline, equivalent
to roughly 900 more airport trips per year in the sample (~1,400 → ~2,300).

Two caveats apply. First, the absolute trip count is small enough that sampling
noise is a real concern — a different random draw from the same year could move
this percentage meaningfully. Second, even at 4.0%, Yellow Taxi handles a
fraction of total airport ground transport. This is a directional recovery
signal, not evidence of a competitive resurgence against TNCs. A borough-level
and route-level cut of the full TLC public dataset would be needed to size the
opportunity reliably.


In [ ]:
airport_q = duckdb.sql('''
    WITH b AS (
        SELECT pickup_year, trip_type, COUNT(*) AS n
        FROM trips_fe WHERE pickup_year < 2023
        GROUP BY pickup_year, trip_type
    ),
    t AS (SELECT pickup_year, SUM(n) AS tot FROM b GROUP BY pickup_year)
    SELECT b.pickup_year, b.trip_type,
           ROUND(100.0 * b.n / t.tot, 2) AS pct
    FROM b JOIN t USING (pickup_year)
    ORDER BY b.pickup_year, b.trip_type
''').pl()

display(airport_q.to_pandas().pivot(
    index='pickup_year', columns='trip_type', values='pct'))

years = sorted(airport_q['pickup_year'].unique().to_list())
airport_pct = [
    float(airport_q.filter(
        (pl.col('pickup_year') == y) & (pl.col('trip_type') == 'Airport')
    ).select('pct').to_series()[0])
    if airport_q.filter(
        (pl.col('pickup_year') == y) & (pl.col('trip_type') == 'Airport')
    ).shape[0] > 0 else 0.0
    for y in years
]

bar_colors = [RED if p == max(airport_pct) else ORANGE for p in airport_pct]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(years, airport_pct, color=bar_colors, alpha=0.85, width=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_ylim(0, max(airport_pct) * 1.5)
ax.set_ylabel('% of trips on airport rate code')
ax.set_xticks(years)
ax.axhline(airport_pct[0], color=GREY, linestyle='--', lw=1.2,
           label=f'2018 baseline ({airport_pct[0]:.1f}%)')
ax.legend(fontsize=9)
ax.set_title('§ 9  Airport Trip Share (JFK + Newark rate codes)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Congestion Surcharge Adoption

New York City's MTA congestion surcharge on Yellow Taxi trips was introduced on
**February 28, 2019** at $2.50 per trip into Manhattan below 96th Street.

The data confirms near-instant adoption:

- **2018**: 0% coverage — fee did not exist
- **2019**: 84.2% — introduced mid-February, so full-year coverage is high
- **2020–2022**: 90–94% — near-complete, with a small exempt fraction
  (outer-borough or airport-rate trips)

The surcharge is now a structural cost built into almost every Manhattan Yellow
Taxi trip, contributing ~$2.34 to the average total by 2020.


In [ ]:
cong_q = duckdb.sql('''
    SELECT
        pickup_year,
        COUNT(*)                                                        AS trips,
        COUNT(*) FILTER (WHERE has_congestion)                          AS with_congestion,
        ROUND(100.0 * COUNT(*) FILTER (WHERE has_congestion) / COUNT(*), 1) AS cong_pct
    FROM trips_fe
    WHERE pickup_year < 2023
    GROUP BY pickup_year
    ORDER BY pickup_year
''').pl()

display(cong_q)

years    = cong_q['pickup_year'].to_list()
cong_pct = cong_q['cong_pct'].to_list()

bar_colors = ['#DDDDDD' if c == 0 else PURPLE for c in cong_pct]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(years, cong_pct, color=bar_colors, alpha=0.9, width=0.6)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_ylim(0, 115)
ax.set_ylabel('% of trips with congestion surcharge > 0')
ax.set_xticks(years)

ax.annotate('Introduced Feb 2019',
    xy=(2019, cong_pct[years.index(2019)]),
    xytext=(2018.1, cong_pct[years.index(2019)] - 35),
    arrowprops=dict(arrowstyle='->', color='grey'),
    fontsize=9, color='grey', ha='center')

ax.set_title('§ 10  Congestion Surcharge Adoption',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 11. Congestion Surcharge — The Policy Tax Hiding in Your Total

§4 showed total cost rose ~$4.89 from 2018 to 2022. This section asks:
**where exactly did that money go?**

The answer: the 2019 congestion surcharge alone accounts for **$2.34** of the
$4.89 increase — 48%. The *entire* rise in base fare over the same five years
adds only $1.38 (28%). Put differently, a rider comparing their 2018 and 2022
receipts would think fares got more expensive, when really a new line-item —
invisible to them as a separate charge — drove nearly half the increase.

The 2019 intra-year chart confirms the exact switch date: January 2019 shows
**0% congestion coverage**, February jumps to **88%** (introduced Feb 28), and
every subsequent month is stable at 91–94%.


In [ ]:
# ── Delta decomposition: 2018 → 2022 ────────────────────────────────────
delta_q = duckdb.sql('''
    WITH y AS (
        SELECT pickup_year,
            AVG(fare_amount)          AS a_fare,
            AVG(congestion_surcharge) AS a_cong,
            AVG(tip_amount)           AS a_tip,
            AVG(extra)                AS a_extra,
            AVG(mta_tax)              AS a_mta,
            AVG(improvement_surcharge) AS a_imp,
            AVG(tolls_amount)         AS a_toll,
            AVG(total_amount)         AS a_total
        FROM trips_fe WHERE pickup_year IN (2018, 2022)
        GROUP BY pickup_year
    )
    SELECT
        ROUND(MAX(CASE WHEN pickup_year=2022 THEN a_fare  END)
            - MAX(CASE WHEN pickup_year=2018 THEN a_fare  END), 2) AS d_fare,
        ROUND(MAX(CASE WHEN pickup_year=2022 THEN a_cong  END)
            - MAX(CASE WHEN pickup_year=2018 THEN a_cong  END), 2) AS d_cong,
        ROUND(MAX(CASE WHEN pickup_year=2022 THEN a_tip   END)
            - MAX(CASE WHEN pickup_year=2018 THEN a_tip   END), 2) AS d_tip,
        ROUND(MAX(CASE WHEN pickup_year=2022 THEN a_extra END)
            - MAX(CASE WHEN pickup_year=2018 THEN a_extra END), 2) AS d_extra,
        ROUND(MAX(CASE WHEN pickup_year=2022 THEN a_mta   END)
            - MAX(CASE WHEN pickup_year=2018 THEN a_mta   END), 2) AS d_mta,
        ROUND(MAX(CASE WHEN pickup_year=2022 THEN a_imp   END)
            - MAX(CASE WHEN pickup_year=2018 THEN a_imp   END), 2) AS d_imp,
        ROUND(MAX(CASE WHEN pickup_year=2022 THEN a_toll  END)
            - MAX(CASE WHEN pickup_year=2018 THEN a_toll  END), 2) AS d_toll,
        ROUND(MAX(CASE WHEN pickup_year=2022 THEN a_total END)
            - MAX(CASE WHEN pickup_year=2018 THEN a_total END), 2) AS d_total,
        MAX(CASE WHEN pickup_year=2018 THEN a_total END)           AS base_2018
    FROM y
''').pl()

display(delta_q)

# ── 2019 monthly adoption ────────────────────────────────────────────────────
cong_2019 = duckdb.sql('''
    SELECT pickup_month,
        ROUND(100.0 * COUNT(*) FILTER (WHERE has_congestion) / COUNT(*), 1) AS pct_with_cong,
        ROUND(AVG(congestion_surcharge), 2)                                  AS avg_surcharge
    FROM trips_fe WHERE pickup_year = 2019
    GROUP BY pickup_month
    ORDER BY pickup_month
''').pl()

# ── Build waterfall / delta bar chart ────────────────────────────────────────
row      = delta_q.row(0, named=True)
base     = float(row['base_2018'])
d_fare   = float(row['d_fare'])
d_cong   = float(row['d_cong'])
d_tip    = float(row['d_tip'])
d_extra  = float(row['d_extra'])
d_other  = float(row['d_mta']) + float(row['d_imp']) + float(row['d_toll'])
d_total  = float(row['d_total'])

components = [
    ('Base\nFare',   d_fare,  BLUE),
    ('Congestion\n(policy)', d_cong,  ORANGE),
    ('Tip',          d_tip,   GREEN),
    ('Extra /\nRush hr',  d_extra, RED),
    ('MTA + Imp\n+ Tolls', d_other, GREY),
]

months     = cong_2019['pickup_month'].to_list()
pct_cong   = cong_2019['pct_with_cong'].to_list()
month_lbl  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── left: delta bar chart with % annotation ───────────────────────────────
ax = axes[0]
labels = [c[0] for c in components]
values = [c[1] for c in components]
colors = [c[2] for c in components]
bars = ax.bar(labels, values, color=colors, alpha=0.88, width=0.6, edgecolor='white')
for bar, v in zip(bars, values):
    pct = 100 * v / d_total
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.04,
            f'+${v:.2f}\n({pct:.0f}% of\ntotal rise)',
            ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.set_title('2018 → 2022: What drove the $4.89 increase?')
ax.set_ylabel('Average dollar change per trip')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylim(-0.1, d_cong * 1.8)

# ── right: 2019 monthly congestion adoption ───────────────────────────────
ax2 = axes[1]
bar_clr = [RED if m == 1 else PURPLE for m in months]
bars2 = ax2.bar([month_lbl[m-1] for m in months], pct_cong,
                color=bar_clr, alpha=0.88, width=0.7)
ax2.bar_label(bars2, fmt='%.0f%%', padding=2, fontsize=8)
ax2.set_ylim(0, 115)
ax2.set_ylabel('% of trips with congestion surcharge')
ax2.set_title('2019: Month-by-Month Adoption')
ax2.annotate('Jan: pre-policy\n(0%)',
    xy=(0, 0.5), xytext=(1.5, 40),
    arrowprops=dict(arrowstyle='->', color='grey'),
    fontsize=8.5, color=RED, ha='center')
ax2.annotate('Feb 28: policy starts\n→ 88% instantly',
    xy=(1, pct_cong[1]), xytext=(4.5, 70),
    arrowprops=dict(arrowstyle='->', color='grey'),
    fontsize=8.5, color='grey', ha='center')

plt.suptitle('§ 11  Congestion Surcharge: The Policy Tax Hiding in Your Total',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 12. Tip Prompt Engineering — How Defaults Reshaped Rider Behavior

The §8 bucket chart showed that tip rates on card trips climbed from ~22% to
~27% median. This section digs into the *mechanism*: in-app prompt architecture.

When taxi apps display a three-button tip prompt, riders are strongly anchored
to the middle option. Changing those defaults from [15 / 20 / 25%] to
[20 / 25 / 30%] — or adding a 30% button — does not require riders to become
more generous. It requires only that they continue clicking the path of least
resistance.

**Two charts reveal this:**

1. The **tip distribution histogram** (card trips, tip > 0) compares 2018 vs
   2022. In 2018 the mass of tips clusters below 25%; by 2022 it has shifted
   almost entirely above 25%, with a notable spike at ~30%.

2. The **year-by-year summary** tracks mean tip rate, median tip rate, and the
   share of card trips that left *any* tip — showing the prompt shift happened
   sharply between 2018 and 2019, exactly when the surcharge and app redesigns
   coincided.


In [ ]:
# ── Histogram distribution of tip_pct (card trips, tip > 0) ────────────────
hist_q = duckdb.sql('''
    WITH c AS (
        SELECT pickup_year, tip_pct
        FROM trips_fe
        WHERE pickup_year IN (2018, 2022) AND payment_type = 1 AND tip_pct > 0
    )
    SELECT pickup_year, ROUND(tip_pct) AS tip_bin, COUNT(*) AS n
    FROM c
    WHERE tip_pct BETWEEN 1 AND 50
    GROUP BY pickup_year, tip_bin
    ORDER BY pickup_year, tip_bin
''').pl()

# ── Year-by-year tip summary ──────────────────────────────────────────────────
tip_yoy = duckdb.sql('''
    SELECT
        pickup_year,
        ROUND(AVG(tip_pct) FILTER (WHERE payment_type=1 AND tip_pct>0), 1)       AS mean_tip_pct,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY tip_pct)
              FILTER (WHERE payment_type=1 AND tip_pct>0), 1)                     AS median_tip_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE payment_type=1 AND tip_pct=0)
            / COUNT(*) FILTER (WHERE payment_type=1), 1)                          AS pct_zero_tip,
        ROUND(AVG(tip_amount) FILTER (WHERE payment_type=1), 2)                   AS avg_tip_dollars
    FROM trips_fe
    WHERE pickup_year < 2023
    GROUP BY pickup_year
    ORDER BY pickup_year
''').pl()

display(tip_yoy)

years = tip_yoy['pickup_year'].to_list()

# ── Build charts ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left: overlaid histogram 2018 vs 2022
ax = axes[0]
for yr, col, alpha, lbl in [(2018, BLUE, 0.65, '2018'), (2022, ORANGE, 0.65, '2022')]:
    sub = hist_q.filter(pl.col('pickup_year') == yr)
    bins = sub['tip_bin'].to_list()
    ns   = sub['n'].to_list()
    total = sum(ns)
    pcts  = [100 * v / total for v in ns]
    ax.bar(bins, pcts, width=0.8, color=col, alpha=alpha, label=lbl)
ax.set_xlim(0, 45)
ax.set_xlabel('Tip % (card trips, tip > 0)')
ax.set_ylabel('% of tipping card trips')
ax.axvline(20, color='grey', linestyle=':', lw=1.2, label='20% reference')
ax.axvline(25, color='black', linestyle=':', lw=1.2, label='25% reference')
ax.axvline(30, color='brown', linestyle=':', lw=1.2, label='30% reference')
ax.legend(fontsize=9)
ax.set_title('Tip Rate Distribution: 2018 vs 2022')

# right: mean + median trend lines
ax2 = axes[1]
ax2.plot(years, tip_yoy['mean_tip_pct'].to_list(),
         'o-', color=ORANGE, lw=2, ms=7, label='Mean tip %')
ax2.plot(years, tip_yoy['median_tip_pct'].to_list(),
         's--', color=BLUE, lw=2, ms=7, label='Median tip %')
ax2.set_ylabel('Tip rate (card trips, tip > 0)')
ax2.set_xticks(years)
ax2.legend(fontsize=10)

ax2b = ax2.twinx()
ax2b.bar(years, tip_yoy['pct_zero_tip'].to_list(),
         alpha=0.25, color=RED, width=0.5, label='Zero-tip card trips (%)')
ax2b.set_ylabel('Zero-tip share of card trips (%)', color=RED)
ax2b.tick_params(axis='y', colors=RED)
ax2b.set_ylim(0, 30)
ax2b.legend(loc='upper right', fontsize=9)

ax2.set_title('Mean / Median Tip % and Zero-Tip Rate by Year')
ax2.set_ylim(18, 32)

plt.suptitle('§ 12  Tip Prompt Engineering: Defaults Shape Rider Behaviour',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 13. The 2020 Speed Anomaly — Rush Hour Won the Race

§5 showed that average trip speed jumped from 11.3 mph to 13.1 mph in 2020.
Two competing explanations exist:

1. **Trip-type shift**: trips in 2020 were shorter / to less congested areas,
   so naturally faster
2. **Congestion collapse**: roads emptied, making *every* trip faster regardless
   of route

The data strongly favours explanation 2. This section shows:

**Speed by hour of day:** In normal years the morning (8–9 am) and evening
(5–7 pm) peaks drag average speed well below 12 mph. In 2020 those same
hours run at 13–15 mph. The improvement is *largest at peak hours*, not at
off-peak hours — exactly the opposite of what a trip-type shift would predict.

**Speed by distance bin:** Controlling for trip distance, 2020 is faster
across *every* distance bucket (< 1 mi, 1–2 mi, 2–3 mi, 3–5 mi, 5+ mi).
The effect is not confined to short or long trips; it is structural.


In [ ]:
# ── Speed by hour of day, all years ──────────────────────────────────────────
speed_hour = duckdb.sql('''
    SELECT pickup_year, pickup_hour,
           ROUND(AVG(speed_mph), 2) AS avg_speed,
           COUNT(*)                 AS trips
    FROM trips_fe
    WHERE pickup_year BETWEEN 2018 AND 2022
    GROUP BY pickup_year, pickup_hour
    ORDER BY pickup_year, pickup_hour
''').pl()

# ── Speed by distance bin, all years ─────────────────────────────────────────
speed_dist = duckdb.sql('''
    WITH binned AS (
        SELECT pickup_year,
            CASE
                WHEN trip_distance <  1 THEN '< 1 mi'
                WHEN trip_distance <  2 THEN '1-2 mi'
                WHEN trip_distance <  3 THEN '2-3 mi'
                WHEN trip_distance <  5 THEN '3-5 mi'
                ELSE                         '5+ mi'
            END AS dist_bin,
            speed_mph
        FROM trips_fe WHERE pickup_year < 2023
    )
    SELECT pickup_year, dist_bin,
           ROUND(AVG(speed_mph), 1) AS avg_speed,
           COUNT(*)                 AS trips
    FROM binned
    GROUP BY pickup_year, dist_bin
    ORDER BY pickup_year,
        CASE dist_bin
            WHEN '< 1 mi' THEN 0 WHEN '1-2 mi' THEN 1 WHEN '2-3 mi' THEN 2
            WHEN '3-5 mi' THEN 3 ELSE 4
        END
''').pl()

display(speed_dist.filter(pl.col('pickup_year').is_in([2018, 2020, 2022])))

# ── Build charts ──────────────────────────────────────────────────────────────
all_years  = [2018, 2019, 2020, 2021, 2022]
year_style = {
    2018: (BLUE,   '-',  1.5, 'o'),
    2019: (GREEN,  '-',  1.5, 's'),
    2020: (RED,    '-',  3.0, 'D'),   # highlighted
    2021: (PURPLE, '--', 1.5, '^'),
    2022: (ORANGE, '-',  1.5, 'v'),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── left: speed by hour ───────────────────────────────────────────────────────
ax = axes[0]
for yr in all_years:
    sub  = speed_hour.filter(pl.col('pickup_year') == yr)
    hrs  = sub['pickup_hour'].to_list()
    spds = sub['avg_speed'].to_list()
    col, ls, lw, mk = year_style[yr]
    ax.plot(hrs, spds, linestyle=ls, lw=lw, color=col,
            marker=mk, ms=4, label=str(yr),
            zorder=5 if yr == 2020 else 2)
ax.set_xlabel('Hour of day (pickup)')
ax.set_ylabel('Avg trip speed (mph)')
ax.set_xticks(range(0, 24, 3))
ax.axvspan(7, 10, alpha=0.07, color='grey', label='AM peak')
ax.axvspan(16, 20, alpha=0.07, color='brown')
ax.set_title('Speed by Hour of Day')
ax.legend(fontsize=9, ncol=2)

# ── right: speed by distance bin (2018 / 2020 / 2022) ────────────────────────
ax2 = axes[1]
plot_years  = [2018, 2020, 2022]
dist_bins   = ['< 1 mi', '1-2 mi', '2-3 mi', '3-5 mi', '5+ mi']
x_pos       = range(len(dist_bins))
bar_w       = 0.25
offsets     = [-bar_w, 0, bar_w]

for yr, off in zip(plot_years, offsets):
    sub    = speed_dist.filter(pl.col('pickup_year') == yr)
    speeds = []
    for db in dist_bins:
        row = sub.filter(pl.col('dist_bin') == db)
        speeds.append(float(row['avg_speed'][0]) if row.shape[0] > 0 else 0.0)
    col = year_style[yr][0]
    bars = ax2.bar([x + off for x in x_pos], speeds,
                   width=bar_w, color=col, alpha=0.85, label=str(yr))
    ax2.bar_label(bars, fmt='%.1f', padding=2, fontsize=7.5)

ax2.set_xticks(list(x_pos))
ax2.set_xticklabels(dist_bins, fontsize=10)
ax2.set_ylabel('Avg trip speed (mph)')
ax2.set_title('Speed by Distance Bin: 2018 / 2020 / 2022')
ax2.legend(fontsize=10)
ax2.set_ylim(0, max(
    [float(speed_dist.filter((pl.col('pickup_year').is_in(plot_years)) & (pl.col('dist_bin') == db))['avg_speed'].max())
     for db in dist_bins]
) * 1.3)

plt.suptitle('§ 13  The 2020 Speed Anomaly: Rush Hour Won the Race',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
